In [1]:
from pprint import pprint

import numpy as np
from f16 import SubsonicF16

import archimedes as arc

%load_ext autoreload
%autoreload 2

In [2]:
from atmosphere import LinearAtmosphere, StandardAtmosphere1976

lin_atm = LinearAtmosphere()
std_atm = StandardAtmosphere1976(
    Rs=1716.3,  # ft·lbf/slug-R
    gamma=1.4,  # [-]
    g0=32.17,  # ft/s²
)

Vt = 500.0  # ft/s
alt = 10000.0  # ft

mach, qbar = lin_atm(Vt, alt)
print("Linear Atmosphere:")
print(f"\tMach: {mach:.2f}, Dynamic Pressure: {qbar:.2f} lbf/ft²")
mach, qbar = std_atm(Vt, alt)
print("Standard Atmosphere 1976:")
print(f"\tMach: {mach:.2f}, Dynamic Pressure: {qbar:.2f} lbf/ft²")

Linear Atmosphere:
	Mach: 0.46, Dynamic Pressure: 219.77 lbf/ft²
Standard Atmosphere 1976:
	Mach: 0.46, Dynamic Pressure: 219.38 lbf/ft²


In [3]:
model = SubsonicF16.from_yaml("config.yaml", "full")
result = model.trim(vt=500.0, turn_rate=0.1, alt=10000.0, gamma=0.0)
pprint(result.variables)
pprint(result.state)
pprint(result.inputs)

x0 = result.state
u0 = result.inputs

# Flatten nested struct -> flat array for scipy
x0_flat, unravel = arc.tree.ravel(x0)


def ode_rhs(t, x_flat, u0):
    x = unravel(x_flat)
    x_t = model.dynamics(t, x, u0)
    x_t_flat, _ = arc.tree.ravel(x_t)
    return x_t_flat


t0, tf = 0.0, 60.0
ts = np.arange(t0, tf, 0.1)
xs_flat = arc.odeint(ode_rhs, (t0, tf), x0_flat, t_eval=ts, args=(u0,))
xs = arc.vmap(unravel)(xs_flat.T)

TrimVariables(alpha=array(0.13016497),
              beta=array(0.00054762),
              throttle=array(0.35652054),
              elevator=array(-4.09466106),
              aileron=array(0.03450114),
              rudder=array(-0.3260409))
State(pos=array([     0.,      0., -10000.]),
      att=EulerAngles([1.00300943 0.0707439  0.        ], seq='xyz'),
      v_B=array([4.95770173e+02, 2.71495945e-01, 6.48988609e+01]),
      w_B=array([-0.00706849,  0.08409843,  0.05364224]),
      eng=NASAEngine.State(power=array(23.15244399)),
      aero=F16Aero.State(),
      elevator=LagActuator.State(position=np.float64(-4.094661055881544)),
      aileron=LagActuator.State(position=np.float64(0.034501137468201326)),
      rudder=LagActuator.State(position=np.float64(-0.326040898652823)))
Input(throttle=array(0.35652054),
      elevator=array(-4.09466106),
      aileron=array(0.03450114),
      rudder=array(-0.3260409))


In [10]:
model.m

636.94267640659

In [9]:
model.J_B

array([[ 9496.,     0.,  -982.],
       [    0., 55814.,     0.],
       [ -982.,     0., 63100.]])

In [8]:
model.hx * 0.0421401

6.742416

In [46]:
@arc.compile
def forward(x0, u0):
    x0_flat, _ = arc.tree.ravel(x0)
    xs_flat = arc.odeint(ode_rhs, (t0, tf), x0_flat, t_eval=ts, args=(u0,))
    xs = arc.vmap(unravel)(xs_flat.T)
    return xs


xs = forward(x0, u0)

In [47]:
%%timeit
xs = forward(x0, u0)  # 60 s / 7.65 ms = 8000x realtime

9.26 ms ± 716 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [49]:
# Codegen

x0_flat, unravel_x = arc.tree.ravel(x0)
u0_flat, unravel_u = arc.tree.ravel(u0)


def f16_ode(t, x_flat, u_flat, p):
    x = unravel_x(x_flat)
    u = unravel_u(u_flat)
    x_t = model.dynamics(t, x, u)
    x_t_flat, _ = arc.tree.ravel(x_t)
    return x_t_flat


f16_dyn = arc.discretize(f16_ode, 0.01, method="rk4")


@arc.compile(name="f16")
def step(t, x, u):
    x_flat, _ = arc.tree.ravel(x)
    u_flat, _ = arc.tree.ravel(u)
    x_new_flat = f16_dyn(t, x_flat, u_flat, None)
    x_new = unravel_x(x_new_flat)
    return x_new


args = (0.0, x0, u0)
return_names = ("x_new",)

arc.codegen(step, args=args, return_names=return_names, output_dir="f16_hil/archimedes")